 ## Уровень Hard — идеалы в $\mathbb Z$ и $K[x]$



## 1. Теория: идеалы и главные идеалы

Пусть $R$ — ассоциативное кольцо с единицей. Подмножество $I \subseteq R$ называется **идеалом**, если выполняются три условия:

1. $0 \in I$;
2. вместе с $a,b \in I$ в $I$ лежит и их разность: $a - b \in I$;
3. вместе с $a \in I$ и произвольным $r \in R$ в $I$ лежит произведение $ra$.

Иначе говоря, идеал — это подгруппа по сложению, замкнутая относительно умножения слева на элементы всего кольца.

Если $S \subseteq R$ — некоторое множество, то идеал, **порождаемый** $S$, обозначается $(S)$ и состоит из всех конечных линейных комбинаций
$$
\sum_{i=1}^n r_i s_i,\qquad r_i \in R,\ s_i \in S.
$$

Важный частный случай — когда $S$ одноэлементно: $S = \{a\}$. Тогда $(a)$ называется **главным идеалом**, порождённым $a$.

Кольцо называется **кольцом главных идеалов (principal ideal domain, PID)**, если любой его идеал является главным. В этой работе нам важны два классических примера PID:

- кольцо целых чисел $\mathbb Z$;
- кольцо многочленов одной переменной $K[x]$ над полем $K$ (например, $K = \mathbb Q$).


### 1.1. НОД как порождающий идеала

В кольцах главных идеалов возникает тесная связь между идеалами и наибольшим общим делителем.

Пусть $R$ — PID и даны элементы $a_1,\dots,a_k \in R$. Тогда идеал
$$
I = (a_1,\dots,a_k)
$$
оказывается равен главному идеалу $(d)$, где $d$ — некоторый элемент, называемый **наибольшим общим делителем** $a_1,\dots,a_k$ (определён с точностью до умножения на обратимый элемент).

Для наших двух колец это означает:

- для целых чисел: $(a_1,\dots,a_k) = (d)$, где $d = \gcd(a_1,\dots,a_k)$ и $d \ge 0$;
- для многочленов: $(f_1,\dots,f_k) = (d(x))$, где $d(x)$ — НОД многочленов, обычно берём его моническим (старший коэффициент $=1$).

Здесь и там НОД можно эффективно найти с помощью **алгоритма Евклида**. Именно его и реализуем в коде.


### 1.2. Как по порождающему проверить принадлежность идеалу

Если $I = (d)$ — главный идеал в PID, то элемент $f \in R$ лежит в $I$ тогда и только тогда, когда $d$ делит $f$:
$$
f \in (d) \quad\Longleftrightarrow\quad \exists r \in R:\ f = rd.
$$

В наших конкретных кольцах это превращается в очень удобные критерии:

- в $\mathbb Z$: $f \in (d)$ тогда и только тогда, когда $d \mid f$ как обычный делитель целых чисел;
- в $K[x]$: $f(x) \in (d(x))$ тогда и только тогда, когда $d(x)$ делит $f(x)$ как многочлен, то есть при делении $f(x)$ на $d(x)$ остаток равен нулю.

Поэтому, если мы умеем:
1. по набору элементов находить их НОД $d$;
2. проверять делимость на $d$,

то умеем отвечать на вопрос «принадлежит ли заданный элемент порождённому идеалу?».

Дальше реализуем эти шаги в коде.


## 2. Алгоритм Евклида в $\mathbb Z$

Алгоритм Евклида основан на следующем наблюдении: если
$$
a = bq + r,\qquad 0 \le r < |b|,
$$
то множества делителей $a$ и $b$ имеют тот же НОД, что и пара $(b, r)$. Поэтому
$$
\gcd(a,b) = \gcd(b,r).
$$

Пошагово:

1. Заменяем пару $(a,b)$ на $(b, a \bmod b)$.
2. Повторяем, пока второй компонент не станет нулём.
3. Последний ненулевой остаток и есть $\gcd(a,b)$.



In [1]:
from typing import List, Union

def gcd_int(a: int, b: int) -> int:
    """
    Классический алгоритм Евклида для целых чисел.

    Параметры
    ----------
    a, b : int
        Два целых числа (могут быть отрицательные).

    Возвращает
    ----------
    int
        НОД(a, b), всегда неотрицательное число.
    """
    # Берём модули, знак для НОД дальше не важен
    a, b = abs(a), abs(b)
    # Пока второй элемент не нулевой, крутим цикл
    while b != 0:
        a, b = b, a % b
    return a


def gcd_int_list(nums: List[int]) -> int:
    """
    НОД нескольких целых чисел.

    Здесь просто свёртка по алгоритму Евклида:
    gcd(a1,...,ak) = gcd(...gcd(gcd(a1,a2),a3)..., ak).
    """
    if not nums:
        raise ValueError("Список чисел пустой, НОД не определён")
    # Нормализуем по модулю
    nums = [abs(int(n)) for n in nums]
    g = 0
    for n in nums:
        g = gcd_int(g, n)
    return g


### 2.1. Пример для целых чисел

Пусть идеал порождён числами $12, 18, 30$:
$$
I = (12,18,30) \subset \mathbb Z.
$$
По теории в $\mathbb Z$ он должен совпадать с идеалом $(d)$, где $d = \gcd(12,18,30)$.

Посчитаем НОД программно.


In [2]:
ints_example = [12, 18, 30]
d = gcd_int_list(ints_example)
print("Порождающие идеала:", ints_example)
print("НОД =", d)

# небольшая проверка "на глаз": все ли a_i делятся на d?
for a in ints_example:
    print(f"{a} делится на {d}? ->", a % d == 0)


Порождающие идеала: [12, 18, 30]
НОД = 6
12 делится на 6? -> True
18 делится на 6? -> True
30 делится на 6? -> True


## 3. Многочлены $K[x]$ и деление с остатком

Теперь переходим к кольцу $K[x]$ многочленов одной переменной. Мы будем работать с коэффициентами в поле рациональных чисел $\mathbb Q$, чтобы избежать ошибок округления. В коде это удобно делать через класс `Fraction` из стандартной библиотеки Python.

Представление многочлена:

- многочлен $a_0 + a_1 x + \dots + a_n x^n$ кодируем списком коэффициентов `[a0, a1, ..., an]`;
- степень многочлена — это индекс последнего ненулевого коэффициента;
- деление с остатком реализуем почти так же, как в обычной алгебре: по старшим членам.

Алгоритм Евклида для многочленов устроен аналогично целочисленному: вместо обычного деления берём деление многочленов с остатком и повторяем шаги до нулевого остатка.


In [3]:
from fractions import Fraction

def to_fraction_list(data: List[Union[int, float, Fraction]]) -> List[Fraction]:
    """Преобразует список чисел в список Fraction (для точной арифметики)."""
    return [c if isinstance(c, Fraction) else Fraction(c) for c in data]

def poly_trim(p: List[Fraction]) -> List[Fraction]:
    """Убирает ведущие нули справа, но оставляет хотя бы один коэффициент."""
    if not p:
        return [Fraction(0)]
    i = len(p) - 1
    while i > 0 and p[i] == 0:
        i -= 1
    return p[: i + 1]

def poly_is_zero(p: List[Fraction]) -> bool:
    """Проверка, что многочлен нулевой."""
    return all(c == 0 for c in p)

def poly_deg(p: List[Fraction]) -> int:
    """Степень многочлена (deg 0 = 0, deg 0-полинома условно не используется)."""
    p = poly_trim(p)
    return len(p) - 1

def poly_divmod(a: List[Fraction], b: List[Fraction]):
    """Деление многочленов a/b над Q. Возвращает пару (q, r) с deg r < deg b."""
    a = poly_trim(a[:])
    b = poly_trim(b[:])
    if poly_is_zero(b):
        raise ZeroDivisionError("деление на нулевой многочлен")
    # если степень делимого меньше, частное = 0, остаток = a
    if poly_deg(a) < poly_deg(b):
        return [Fraction(0)], a
    m = poly_deg(a)
    n = poly_deg(b)
    q = [Fraction(0) for _ in range(m - n + 1)]
    # классический цикл деления по старшим членам
    while (not poly_is_zero(a)) and poly_deg(a) >= n:
        da = poly_deg(a)
        coef = a[da] / b[n]     # сколько раз старший член b помещается в старший член a
        shift = da - n          # на сколько степеней x нужно сдвинуть
        q[shift] += coef
        # вычитаем coef * x^shift * b из текущего остатка
        for i in range(n + 1):
            a[shift + i] -= coef * b[i]
        a = poly_trim(a)
    return poly_trim(q), poly_trim(a)

def poly_monic(p: List[Fraction]) -> List[Fraction]:
    """Нормирует многочлен так, чтобы старший коэффициент был равен 1."""
    p = poly_trim(p)
    if poly_is_zero(p):
        return [Fraction(0)]
    lc = p[-1]
    if lc == 1:
        return p
    return [c / lc for c in p]

def poly_gcd(a: List[Fraction], b: List[Fraction]) -> List[Fraction]:
    """НОД многочленов над Q. Результат делаем моническим."""
    a = poly_trim(a[:])
    b = poly_trim(b[:])
    if poly_is_zero(a):
        return poly_monic(b)
    if poly_is_zero(b):
        return poly_monic(a)
    # алгоритм Евклида: пока второй многочлен не нулевой
    while not poly_is_zero(b):
        _, r = poly_divmod(a, b)
        a, b = b, r
    return poly_monic(a)


### 3.1. Пример НОД многочленов

Возьмём два многочлена
$$
f_1(x) = x^3 - x = x(x-1)(x+1),\qquad
f_2(x) = x^2 - 1 = (x-1)(x+1).
$$

Их НОД очевиден из факторизации: $d(x) = x^2 - 1$. Проверим это с помощью написанных функций.


In [4]:
# f1(x) = x^3 - x
f1 = to_fraction_list([0, -1, 0, 1])
# f2(x) = x^2 - 1
f2 = to_fraction_list([-1, 0, 1])

g = poly_gcd(f1, f2)
print("НОД(f1, f2) =", g)

q1, r1 = poly_divmod(f1, g)
q2, r2 = poly_divmod(f2, g)
print("Проверка делимости:")
print("f1 = q1 * g + r1, r1 =", r1)
print("f2 = q2 * g + r2, r2 =", r2)


НОД(f1, f2) = [Fraction(-1, 1), Fraction(0, 1), Fraction(1, 1)]
Проверка делимости:
f1 = q1 * g + r1, r1 = [Fraction(0, 1)]
f2 = q2 * g + r2, r2 = [Fraction(0, 1)]


## 4. Класс `RingElement` и функция `gcd_ring_elements`

По условию уровня Hard предлагается общий интерфейс для работы с элементами кольца: класс `RingElement`, который умеет хранить либо целое число, либо список коэффициентов многочлена. На базе этого класса нужно написать функцию

```python
gcd_ring_elements(elements: List[RingElement]) -> RingElement
```

которая возвращает порождающий главного идеала, задаваемого элементами `elements`:

- в случае $\mathbb Z$ — обычный НОД чисел;
- в случае $K[x]$ — монический НОД многочленов.


In [ ]:
class RingElement:
    """Базовый класс для элементов колец: целых чисел и полиномов."""

    def __init__(self, data: Union[int, List[Union[int, float, Fraction]]]):
        """
        Параметр `data` интерпретируется так:
        - если это `int` → элемент кольца ℤ;
        - если это список `[a0, a1, ..., an]` → многочлен a0 + a1*x + ... + an*x^n из K[x].

        Чтобы не усложнять, считаем, что коэффициенты лежат в Q (используем Fraction).
        """
        self.data = data
        self.is_polynomial = isinstance(data, list)

    def __repr__(self) -> str:
        """Чуть более человекочитаемое представление элемента."""
        if self.is_polynomial:
            coeffs = poly_trim(to_fraction_list(self.data))
            terms = []
            for i, c in enumerate(coeffs):
                if c == 0:
                    continue
                # печатаем целые коеффы без дробей
                if isinstance(c, Fraction) and c.denominator == 1:
                    c_str = str(c.numerator)
                else:
                    c_str = str(c)
                if i == 0:
                    terms.append(f"{c_str}")
                elif i == 1:
                    terms.append(f"{c_str}*x")
                else:
                    terms.append(f"{c_str}*x^{i}")
            return " + ".join(terms) if terms else "0"
        else:
            return str(self.data)


def gcd_ring_elements(elements: List[RingElement]) -> RingElement:
    """
    Возвращает порождающий главный идеал, порождённый заданными элементами.

    * Для ℤ: это НОД целых чисел (берём неотрицательным).
    * Для K[x]: монический НОД многочленов (коэффициенты рассматриваем в Q).

    Если все переданные элементы нулевые, возвращаем нулевой элемент (идеал {0}).
    """
    if not elements:
        raise ValueError("Нужен хотя бы один элемент, иначе идеал не задан.")

    is_poly = elements[0].is_polynomial
    # легкая защита от "смешивания миров"
    if any(el.is_polynomial != is_poly for el in elements):
        raise ValueError("Нельзя смешивать целые числа и многочлены в одном идеале.")

    if not is_poly:
        # случай ℤ
        nums = [int(el.data) for el in elements]
        if all(n == 0 for n in nums):
            return RingElement(0)
        d = gcd_int_list(nums)
        return RingElement(d)
    else:
        # случай многочленов
        polys: List[List[Fraction]] = []
        for el in elements:
            coeffs = el.data
            if not isinstance(coeffs, list):
                raise TypeError("Для полинома ожидается список коэффициентов.")
            p = poly_trim(to_fraction_list(coeffs))
            if not poly_is_zero(p):
                polys.append(p)
        if not polys:
            # все порождающие были нулевые
            return RingElement([0])

        g = polys[0]
        for p in polys[1:]:
            g = poly_gcd(g, p)

        # Переводим Fractions обратно в привычный формат (инты, если возможно)
        result_coeffs: List[Union[int, float]] = []
        g = poly_trim(g)
        for c in g:
            if isinstance(c, Fraction) and c.denominator == 1:
                result_coeffs.append(int(c.numerator))
            else:
                result_coeffs.append(float(c))
        return RingElement(result_coeffs)


def is_in_principal_ideal(element: RingElement, generators: List[RingElement]) -> bool:
    """
    Проверка, лежит ли элемент `element` в идеале, порождённом списком `generators`.

    С точки зрения теории:
    1. Находим d = gcd(generators).
    2. Проверяем, делится ли element на d.

    Работает и в ℤ, и в K[x] (коэффициенты в Q).
    """
    if not generators:
        # идеал ( ) интерпретируем как {0}
        if element.is_polynomial:
            return False
        return int(element.data) == 0

    is_poly = generators[0].is_polynomial
    if element.is_polynomial != is_poly:
        raise ValueError("Элемент и порождающие должны быть из одного кольца.")

    d = gcd_ring_elements(generators)

    if not is_poly:
        d_val = int(d.data)
        e_val = int(element.data)
        if d_val == 0:
            # идеал {0}
            return e_val == 0
        return e_val % d_val == 0
    else:
        gen_coeffs = poly_trim(to_fraction_list(d.data))
        if poly_is_zero(gen_coeffs):
            # идеал {0}
            return poly_is_zero(poly_trim(to_fraction_list(element.data)))
        elem_coeffs = poly_trim(to_fraction_list(element.data))
        _, r = poly_divmod(elem_coeffs, gen_coeffs)
        return poly_is_zero(r)


### 4.1. Пример: целые числа

Пусть
$$
I = (12,18,30) \subset \mathbb Z.
$$
Посмотрим, что вернёт `gcd_ring_elements`, и проверим принадлежность нескольких элементов этому идеалу.


In [6]:
int_elements = [RingElement(12), RingElement(18), RingElement(30)]
d_int = gcd_ring_elements(int_elements)

print("Порождающие:", int_elements)
print("Порождающий главного идеала d =", d_int)

for value in [24, 25, 42]:
    elem = RingElement(value)
    print(f"{value} ∈ (12,18,30)? ->", is_in_principal_ideal(elem, int_elements))


Порождающие: [12, 18, 30]
Порождающий главного идеала d = 6
24 ∈ (12,18,30)? -> True
25 ∈ (12,18,30)? -> False
42 ∈ (12,18,30)? -> True


### 4.2. Пример: многочлены

Возьмём те же многочлены, что и раньше,
$$
f_1(x) = x^3 - x,\qquad f_2(x) = x^2 - 1,
$$
и посмотрим, какой порождающий идеала $(f_1,f_2)$ найдёт наша общая функция. Заодно проверим, лежит ли, например, многочлен $x^4 - 1$ в этом идеале.


In [7]:
f1_elem = RingElement([0, -1, 0, 1])   # x^3 - x
f2_elem = RingElement([-1, 0, 1])      # x^2 - 1
poly_generators = [f1_elem, f2_elem]

d_poly = gcd_ring_elements(poly_generators)
print("Порождающие полиномы:", poly_generators)
print("Порождающий идеала d(x) =", d_poly)

# проверим несколько многочленов на принадлежность
candidates = {
    "x^4 - 1":       RingElement([-1, 0, 0, 0, 1]),
    "x^2 + 1":       RingElement([1, 0, 1]),
    "(x^2 - 1)^2":   RingElement([1, 0, -2, 0, 1]),  # развёрнутая форма
}
for name, elem in candidates.items():
    inside = is_in_principal_ideal(elem, poly_generators)
    print(f"{name} ∈ (f1, f2)? ->", inside)


Порождающие полиномы: [-1*x + 1*x^3, -1 + 1*x^2]
Порождающий идеала d(x) = -1 + 1*x^2
x^4 - 1 ∈ (f1, f2)? -> True
x^2 + 1 ∈ (f1, f2)? -> False
(x^2 - 1)^2 ∈ (f1, f2)? -> True


## 5. Выводы по уровню Hard

1. В кольцах $\mathbb Z$ и $K[x]$ любой идеал главный, поэтому достаточно найти **один** порождающий элемент — НОД заданных элементов.
2. Алгоритм Евклида (для целых и для многочленов) позволяет эффективно вычислять этот НОД и, следовательно, находить порождающий элемента идеала $(a_1,\dots,a_k)$.
3. Зная порождающий $d$, проверка принадлежности элемента идеалу сводится к простой проверке делимости:
   - в $\mathbb Z$ — через остаток от деления;
   - в $K[x]$ — через деление многочленов с остатком.
4. Общая функция `gcd_ring_elements` прячет внутри себя детали реализации алгоритма Евклида и работает сразу в двух «мирах» — для целых и для многочленов.

